# Projeto Extensionista - Dashboard de Enchentes em Porto Alegre - INCT SiMAI - FASE 2
**Nomes:** Amanda Flores Domeneghini, Milena Fontana Bregalda, Mariana Nunes dos Santos, Karen Cristiane Nascimento


**Disciplina**: Coleta, Preparação e Análise de Dados


**Turma**: 310/110

**Professora**: Katherine Bianchini Esper


**Data**: 09/11/2026

Codigo carrega arquivo em formato longo:

In [3]:
# Do enunciado do trabalho:
import json
import pandas as pd

with open("../dados/originais/data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {
        "nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
        "bairro": bairro, 
        **valores
    }
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])

print(df.shape)  # deve dar (6674, 7)

# Extra: cria arquivo em formato json dos dados longos
df.to_json(
    "../dados/processados/data_inctlongo.json", 
    orient="records", 
    indent=2, 
    force_ascii=False
)

(6674, 7)


Analise das inconsistências de forma automatizada:

Inconsistências verificadas:
- Diminuição de prédios afetados conforme o nível do Guaíba aumenta
- Coerência do percentual de prédios afetados com os demais campos
- Coerência do cálculo da população afetada
- Valores ausentes
- Valores negativos
- Registros duplicados de bairro e nível
- Quantidade de prédios afetados maior que o total de prédios existentes
- Níveis de água ausentes ou inesperados
- Presença de todos os bairros nos diferentes níveis

In [4]:
import json
import pandas as pd
import numpy as np

with open("../dados/processados/data_inctlongo.json", "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)


# Transforma os dados em um DataFrame (tipo planilha do Pandas):
df = pd.DataFrame(dados)

print("Dimensões do DataFrame:", df.shape)

print("\nPrimeiras linhas do dataframe:")
print(df.head())


colunas = [
    "total_buildings",
    "affected_buildings",
    "percentage",
    "population",
    "affected_population"
]


# VERIFICAÇÃO 1: Diminuição de prédios afetados conforme o nível do Guaíba aumenta
# Faz isso ordenando os dados por bairro e nível do rio, calcula a diferença de prédios afetados
# de um nível para o próximo e filtra apenas os casos em que essa variação é negativa

print("\n---Diminuição de prédios afetados conforme o nível do Guaíba aumenta---")

df_ordenado = df.sort_values(["bairro", "nivel_m"]).copy()

df_ordenado["variacao_predios"] = (
    df_ordenado.groupby("bairro")["affected_buildings"].diff()
)

diminuicoes = df_ordenado[
    df_ordenado["variacao_predios"] < 0
]

print(diminuicoes[
    ["bairro", "nivel_m", "affected_buildings", "variacao_predios"]
])


# VERIFICAÇÃO 2: Coerência do percentual de prédios afetados com os demais campos
# dividindo os prédios atingidos pelo total, arredonda o valor pra cima e compara com a porcentagem
# informada no arquivo original para filtrar os registros que estao diferentes

print("\n---Coerência do percentual de prédios afetados com os demais campos---")

df["percentage_calculada"] = np.ceil(
    df["affected_buildings"] / df["total_buildings"] * 100
)

percentuais_invalidos = df[
    df["percentage"] != df["percentage_calculada"]
]

print(percentuais_invalidos[
    ["bairro", "nivel_m", "total_buildings",
     "affected_buildings", "percentage", "percentage_calculada"]
])


# VERIFICAÇÃO 3: Coerência do cálculo da população afetada
# multiplicando a população do bairro pela porcentagem afetada, arredonda o valor pra cima e 
# compara ele com o dado informado no arquivo original para filtrar as divergências

print("\n---Coerência do cálculo da população afetada---")

df["population_calculada"] = np.ceil(
    df["population"] * df["percentage"] / 100
)

populacao_invalida = df[
    df["affected_population"] != df["population_calculada"]
]

print(populacao_invalida[
    ["bairro", "nivel_m", "population",
     "percentage", "affected_population", "population_calculada"]
])


# VERIFICAÇÃO 4: Valores ausentes

print("\n---VALORES AUSENTES---")

ausentes = df[colunas].isnull().sum()  # Verifica a quantidade de valores nulos

print(ausentes)


# VERIFICAÇÃO 5: Valores negativos

print("\n---VALORES NEGATIVOS---")

negativos = df[
    (df[colunas] < 0).any(axis=1)  # Verifica se existe algum número negativo
]

print(negativos[["bairro", "nivel_m"] + colunas]) # E se tiver imprime


# VERIFICAÇÃO 6: Registros duplicados de bairro e nível
# faz combinação de bairro e nível do rio, ordena o resultado e imprime
# quais são as duplicatas encontradas junto com a sua quantidade

print("\n---Registros duplicados de bairro e nível---")

duplicados = df[
    df.duplicated(subset=["bairro", "nivel_m"], keep=False)
].sort_values(["bairro", "nivel_m"])

print(duplicados[["bairro", "nivel_m"]])

print("Quantidade de registros duplicados:", len(duplicados))


# VERIFICAÇÃO 7: Quantidade de prédios afetados maior que o total de prédios existentes
# Encontra se eh maior comparando, e exibe os que sao

print("\n---Quantidade de prédios afetados maior que o total de prédios existentes---")

predios_invalidos = df[
    df["affected_buildings"] > df["total_buildings"]
]

print(predios_invalidos[
    ["bairro", "nivel_m", "total_buildings", "affected_buildings"]
])


# VERIFICAÇÃO 8: Níveis de água ausentes ou inesperados
# compara a sequência encontrada com esperados, ausentes e extras
# identificando se tem degraus de altura ausentes ou a mais

print("\n---Níveis de água ausentes ou inesperados---")

niveis_encontrados = sorted(df["nivel_m"].unique())

niveis_esperados = np.round(
    np.arange(3.0, 10.0 + 0.1, 0.1), 1
)

niveis_ausentes = sorted(
    set(niveis_esperados) - set(niveis_encontrados)
)

niveis_extras = sorted(
    set(niveis_encontrados) - set(niveis_esperados)
)

print("Quantidade de níveis encontrados:", len(niveis_encontrados))
print("Quantidade de níveis esperados:", len(niveis_esperados))
print("Níveis ausentes:", niveis_ausentes)
print("Níveis extras:", niveis_extras)

if len(niveis_ausentes) == 0 and len(niveis_extras) == 0:
    print("A sequência de níveis está completa, tudo ok")
else:
    print("Foram encontradas inconsistências na sequência de níveis")


# VERIFICAÇÃO 9: Presença de todos os bairros nos diferentes níveis
# mapeia todos os bairros por nivel, junta em lista completa de bairros da cidade,
# verifica se algum nivel falta algum bairro

print("\n---Presença de todos os bairros nos diferentes níveis---")

bairros_por_nivel = {
    nivel: set(grupo["bairro"])
    for nivel, grupo in df.groupby("nivel_m")
}

todos_bairros = set.union(*bairros_por_nivel.values())

for nivel, bairros in bairros_por_nivel.items():
    ausentes_nivel = todos_bairros - bairros

    if ausentes_nivel:
        print(f"Nível {nivel}: bairros ausentes: {sorted(ausentes_nivel)}")


# MOSTRA O RESUMO FINAL DA PROCURA DE INCONSISTÊNCIAS

print("\n--- RESUMO DA PROCURA DE INCONSISTÊNCIAS ---")

print("Total de registros encontrados:", len(df))
print("Quantidade de níveis:", len(niveis_encontrados))
print("Níveis ausentes:", len(niveis_ausentes))
print("Níveis extras:", len(niveis_extras))
print("Valores ausentes:", ausentes.sum())
print("Registros com valores negativos:", len(negativos))
print("Registros duplicados:", len(duplicados))
print("Registros com prédios afetados acima do total:", len(predios_invalidos))
print("Registros com diminuição de prédios afetados:", len(diminuicoes))
print("Percentuais inconsistentes:", len(percentuais_invalidos))
print("Populações afetadas inconsistentes:", len(populacao_invalida))

print("\nVerificações concluídas!")


Dimensões do DataFrame: (6674, 7)

Primeiras linhas do dataframe:
   nivel_m             bairro  affected_buildings  total_buildings  \
0      3.0  Aberta dos Morros                   0             4696   
1      3.0          Agronomia                   0             1771   
2      3.0           Anchieta                  28              589   
3      3.0        Arquipélago                4685             4688   
4      3.0        Auxiliadora                   0             1995   

   percentage  population  affected_population  
0           0        9854                    0  
1           0        2677                    0  
2           5         791                   40  
3         100        6411                 6411  
4           0        8909                    0  

---Diminuição de prédios afetados conforme o nível do Guaíba aumenta---
                bairro  nivel_m  affected_buildings  variacao_predios
97         Arquipélago      3.1                4680              -5.0
567   

Limpeza dos dados de acordo com as inconsistências encontradas acima:

In [5]:
import json
import pandas as pd
import numpy as np

with open("../dados/processados/data_inctlongo.json", "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

df = pd.DataFrame(dados)

df_limpo = df.sort_values(["bairro", "nivel_m"]).copy() # Ordenados por bairro e nivel do rio

# Nova coluna (affected_buildings_corrigido), maior numero de predios por bairro eh o valor correto
df_limpo["affected_buildings_corrigido"] = (
    df_limpo.groupby("bairro")["affected_buildings"].cummax() # cummax - máximo acumulado
)

# Salva as inconsistencias encontradas
inconsistencias = df_limpo[
    df_limpo["affected_buildings"] != df_limpo["affected_buildings_corrigido"]
]

print("\n---RELATORIO - REGISTROS COM CORRECOES FEITAS---")

# Mostra as inconsistencias
print(inconsistencias[
    ["bairro", "nivel_m", "affected_buildings",
     "affected_buildings_corrigido"]
])

print("\nQuantidade de registros corrigidos:", len(inconsistencias))

# Arruma as inconsistencias
df_limpo["affected_buildings"] = (
    df_limpo["affected_buildings_corrigido"]
)

# Remove a coluna temporaria que serviu pra salvar os dados corrigidos
df_limpo = df_limpo.drop(columns=["affected_buildings_corrigido"])


# Recalcula o percentual com base na quantidade corrigida, ja que a quantidade mudou
df_limpo["percentage"] = np.ceil(
    df_limpo["affected_buildings"] /
    df_limpo["total_buildings"] * 100
)

# Recalcula a população afetada com o percentual que foi recalculado acima
df_limpo["affected_population"] = np.ceil(
    df_limpo["population"] * df_limpo["percentage"] / 100
)

# Salva dados limpados no json novo:
dados_limpos = df_limpo.to_dict(orient="records") # records - pra lista de dicionarios

with open("../dados/processados/data_inctlongoelimpo.json", "w", encoding="utf-8") as arquivo:
    json.dump(dados_limpos, arquivo, ensure_ascii=False, indent=4)

print("\nDimensões após a limpeza:", df_limpo.shape)
print("Dados limpos salvos com sucesso!")



---RELATORIO - REGISTROS COM CORRECOES FEITAS---
              bairro  nivel_m  affected_buildings  \
97       Arquipélago      3.1                4680   
191      Arquipélago      3.2                4680   
285      Arquipélago      3.3                4680   
379      Arquipélago      3.4                4680   
567      Arquipélago      3.6                4680   
...              ...      ...                 ...   
6417  Espírito Santo      9.8                 129   
6511  Espírito Santo      9.9                 131   
6605  Espírito Santo     10.0                 133   
4842         Lageado      8.1                   0   
5522     Rubem Berta      8.8                   0   

      affected_buildings_corrigido  
97                            4685  
191                           4685  
285                           4685  
379                           4685  
567                           4685  
...                            ...  
6417                          2101  
6511             